In [7]:
import os
import polars as pl
import altair as alt

In [8]:
pl.Config(tbl_rows=100)
pl.Config(fmt_str_lengths=150)
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [9]:
df = pl.read_parquet(
    "data/segments.parquet"
).sort(
    ["id", "date"]
).with_columns(
    days_diff = pl.col("date").diff().dt.total_days().over("id"),
    effort_diff = pl.col("effort_count").diff().over("id")
).with_columns(
    avg_daily_efforts = pl.col("effort_diff") / pl.col("days_diff")
).with_columns(
    pl.when(pl.col("city").str.contains("Praha") | pl.col("city").str.contains("Prague")).then(pl.lit('Praha')).when(pl.col('city').str.contains('Brno')).then(pl.lit('Brno')).otherwise(pl.col('city')).alias('city'),
    pl.when(pl.col("effort_diff") < 0).then(pl.lit(0)).otherwise(pl.col("effort_diff")).alias("effort_diff")
).filter(
    pl.col("date").dt.year() == 2025
)

In [10]:
df.filter(pl.col("activity_type") == "Run").group_by("name").agg(pl.col("effort_diff").sum()).sort(by="effort_diff",descending=True).head(100)

name,effort_diff
str,i64
"""Overtake the Boris Bikes""",146886
"""Šlechtovka - finální rovinka""",37576
"""VUT horní dráha 400m""",37557
"""Stadion Na Děkance""",35728
"""Lužánky rovinka""",32983
"""Pražačka 320""",31963
"""Viveros-Paseo alameda""",20790
"""TJ Sokol okruh""",18869
"""Štvanická lávka (do Karlína)""",17650


In [11]:
df.filter(pl.col("activity_type") == "Ride").group_by("name").agg(pl.col("effort_diff").sum()).sort(by="effort_diff",descending=True).head(100)

name,effort_diff
str,i64
"""Richmond ITT1""",553593
"""Die Eine Runde""",108727
"""Box Hill 2.2k""",77164
"""Opičí Časovka, Podolská vodárna - Dvorce""",46214
"""Sa Calobra - Coll dels Reis""",44018
"""Zweite hup u ZOO""",41919
"""Třebešín-dráha K""",34233
"""Zbraslav - Komořany""",32682
"""Alpe d'Huez""",31091


In [12]:
df.filter(pl.col("date").dt.year() == 2025).filter(pl.col("name") == "tenis 1K")

id,name,activity_type,distance,average_grade,elevation_high,elevation_low,total_elevation_gain,start_latlng,end_latlng,country,state,city,effort_count,athlete_count,date,start_to_finish_distance,days_diff,effort_diff,avg_daily_efforts
i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,str,str,i64,i64,"datetime[μs, CET]",f64,i64,i64,f64
8716919,"""tenis 1K""","""Run""",1002.8,-0.3,210.6,205.8,0.0,"[49.20352, 16.56572]","[49.210251, 16.557624]","""Czechia""","""South Moravian Region""","""Brno""",48879,2461,2025-01-01 23:05:20 CET,0.951894,0,30,inf
8716919,"""tenis 1K""","""Run""",1002.8,-0.3,210.6,205.8,0.0,"[49.20352, 16.56572]","[49.210251, 16.557624]","""Czechia""","""South Moravian Region""","""Brno""",48898,2463,2025-01-02 23:22:02 CET,0.951894,1,19,19.0
8716919,"""tenis 1K""","""Run""",1002.8,-0.3,210.6,205.8,0.0,"[49.20352, 16.56572]","[49.210251, 16.557624]","""Czechia""","""South Moravian Region""","""Brno""",48917,2467,2025-01-03 23:21:39 CET,0.951894,0,19,inf
8716919,"""tenis 1K""","""Run""",1002.8,-0.3,210.6,205.8,0.0,"[49.20352, 16.56572]","[49.210251, 16.557624]","""Czechia""","""South Moravian Region""","""Brno""",48945,2467,2025-01-04 23:22:04 CET,0.951894,1,28,28.0
8716919,"""tenis 1K""","""Run""",1002.8,-0.3,210.6,205.8,0.0,"[49.20352, 16.56572]","[49.210251, 16.557624]","""Czechia""","""South Moravian Region""","""Brno""",48994,2479,2025-01-05 23:22:06 CET,0.951894,1,49,49.0
8716919,"""tenis 1K""","""Run""",1002.8,-0.3,210.6,205.8,0.0,"[49.20352, 16.56572]","[49.210251, 16.557624]","""Czechia""","""South Moravian Region""","""Brno""",49020,2480,2025-01-06 23:22:12 CET,0.951894,1,26,26.0
8716919,"""tenis 1K""","""Run""",1002.8,-0.3,210.6,205.8,0.0,"[49.20352, 16.56572]","[49.210251, 16.557624]","""Czechia""","""South Moravian Region""","""Brno""",49063,2481,2025-01-07 23:22:01 CET,0.951894,0,43,inf
8716919,"""tenis 1K""","""Run""",1002.8,-0.3,210.6,205.8,0.0,"[49.20352, 16.56572]","[49.210251, 16.557624]","""Czechia""","""South Moravian Region""","""Brno""",49108,2488,2025-01-08 23:38:16 CET,0.951894,1,45,45.0
8716919,"""tenis 1K""","""Run""",1002.8,-0.3,210.6,205.8,0.0,"[49.20352, 16.56572]","[49.210251, 16.557624]","""Czechia""","""South Moravian Region""","""Brno""",49152,2488,2025-01-09 23:05:30 CET,0.951894,0,44,inf


In [13]:
alt.Chart(
    df.filter(
        pl.col("date").dt.year() == 2025
    ).filter(
        pl.col("name") == "Štvanická lávka (do Karlína)"
    ).with_columns(
        pl.col("date").dt.week().alias("tyden")
    ).group_by(
        "tyden"
    ).agg(
        pl.col('effort_diff').sum()
    ),
    width=700,
    height=300
).mark_bar(
).encode(
    alt.X("tyden:N"),
    alt.Y("effort_diff:Q")
)

alt.Chart(...)

In [14]:
def serie(segment):

    print(f"Polovina pokusů na segmentu {segment}:")
    
    return df.filter(
        pl.col("date").dt.year() == 2025
    ).filter(
        pl.col("name") == segment
    ).with_columns(
        pl.col("date").dt.week().alias("tyden")
    ).group_by(
        "tyden"
    ).agg(
        pl.col('effort_diff').sum()
    ).sort(
    by='tyden'
    )

In [15]:
def mark_shortest_half_span(df: pl.DataFrame, value_col: str = "effort_diff", part: int = 2) -> pl.DataFrame:
    values = df[value_col].to_list()
    n = len(values)
    total = sum(values)
    target = total / part

    best_start, best_end = 0, n  # best window (exclusive end)
    window_sum = 0
    left = 0

    for right in range(n):
        window_sum += values[right]
        # Shrink from left as long as we still meet the target
        while window_sum - values[left] >= target:
            window_sum -= values[left]
            left += 1
        # Now check if current window meets target
        if window_sum >= target:
            if (right - left + 1) < (best_end - best_start):
                best_start, best_end = left, right + 1

    half_col = [best_start <= i < best_end for i in range(n)]
    
    return df.with_columns(pl.Series("half", half_col))


In [16]:
import altair as alt

In [17]:
alt.Chart(
    mark_shortest_half_span(df=serie(segment="Šlechtovka - finální rovinka")),
    width=700,
    height=300
).mark_bar(
).encode(
    alt.X("tyden"),
    alt.Y("effort_diff"),
    alt.Color("half")
)

Polovina pokusů na segmentu Šlechtovka - finální rovinka:


alt.Chart(...)

In [18]:
alt.Chart(
    mark_shortest_half_span(df=serie(segment="Sptint pod Býčí skálou"), part=2),
    width=700,
    height=300
).mark_bar(
).encode(
    alt.X("tyden"),
    alt.Y("effort_diff"),
    alt.Color("half")
)

Polovina pokusů na segmentu Sptint pod Býčí skálou:


alt.Chart(...)

In [19]:
alt.Chart(
    mark_shortest_half_span(df=serie(segment="Lysá hora - 5 km")),
    width=700,
    height=300
).mark_bar(
).encode(
    alt.X("tyden"),
    alt.Y("effort_diff"),
    alt.Color("half")
)

Polovina pokusů na segmentu Lysá hora - 5 km:


alt.Chart(...)

In [20]:
alt.Chart(
    mark_shortest_half_span(df=serie(segment="Adamaov climb")),
    width=700,
    height=300
).mark_bar(
).encode(
    alt.X("tyden"),
    alt.Y("effort_diff"),
    alt.Color("half")
)

Polovina pokusů na segmentu Adamaov climb:


alt.Chart(...)

In [21]:
alt.Chart(
    mark_shortest_half_span(df=serie(segment="Od posledního tunelu do Bílovic")),
    width=700,
    height=300
).mark_bar(
).encode(
    alt.X("tyden"),
    alt.Y("effort_diff"),
    alt.Color("half")
)

Polovina pokusů na segmentu Od posledního tunelu do Bílovic:


alt.Chart(...)

In [22]:
alt.Chart(
    mark_shortest_half_span(df=serie(segment="Bečva - Pustevny Uphill")),
    width=700,
    height=300
).mark_bar(
).encode(
    alt.X("tyden"),
    alt.Y("effort_diff"),
    alt.Color("half")
)

Polovina pokusů na segmentu Bečva - Pustevny Uphill:


alt.Chart(...)